# Tight detector priors (step2)
Compares the step2 fit with the SAY likelihood, the Poisson likelihood, and the Poisson likelihood with 5x tighter priors on the
SnowStorm detector parameters, for the 1D profile scans and for the data flavor triangle.

| scan | likelihood | prior widths dom_eff, ice_abs, ice_scat, holep0, holep1, crystal |
|---|---|---|
| `nbestfit50_all_param_2sigma_1D_10steps_PassingHESE` / `nbestfit50_fine_{0,1,2}` | SAY | 0.1, 0.05, 0.05, 0.5, 0.05, 0.1 |
| `..._PassingHESE_Poisson` / `nbestfit50_Poisson` | Poisson | 0.1, 0.05, 0.05, 0.5, 0.05, 0.1 |
| `..._PassingHESE_Poisson_tightpriors` / `nbestfit50_Poisson_tightpriors` | Poisson | 0.02, 0.01, 0.01, 0.1, 0.01, 0.02 |

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import NNMFit

STEP2_DIR = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor_reunblind"
sys.path.insert(0, STEP2_DIR)
from analysis_utils import (get_1sigma, load_triangle_scan, combine_scans, flavor_fractions,
                            latex_comparison_table, PARAMETERS)

plotting_path = f"{STEP2_DIR}/studies/plots_compare_tight_priors"
os.makedirs(plotting_path, exist_ok=True)

In [ ]:
import matplotlib.font_manager as font_manager
font_axis_label = {'family': 'serif',
        'color':  'black',
        'weight': 'normal',
        'size': 22,
        }
font_title = {'family': 'serif',
        'color':  'black',
        'weight': 'bold',
        'size': 20,
        }
font_legend = font_manager.FontProperties(family='serif',
                                   weight='normal',
                                   style='normal', size=13)

plt.rcParams["figure.figsize"] = (6,6)
plt.rcParams.update({'font.family':'serif'})

## 1D profile scans

In [ ]:
data_fits = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor_reunblind/data_fits"
scan_dirs = {
    "SAY":           f"{data_fits}/nbestfit50_all_param_2sigma_1D_10steps_PassingHESE",
    "Poisson":       f"{data_fits}/nbestfit50_all_param_2sigma_1D_10steps_PassingHESE_Poisson",
    "Poisson tight": f"{data_fits}/nbestfit50_all_param_2sigma_1D_10steps_PassingHESE_Poisson_tightpriors",
}
colors = {"SAY": "steelblue", "Poisson": "darkorange", "Poisson tight": "green"}

drop_unsuccessful = True  # as in fit_result.ipynb
scan_dict = {name: NNMFit.utilities.ScanHandler(d, dump=False, drop_unsuccessful=drop_unsuccessful) for name, d in scan_dirs.items()}
bestfits = {name: scan.get_freefit(drop_unsuccessful="True") for name, scan in scan_dict.items()}

# SAY and Poisson llh values are not comparable; Poisson and Poisson tight differ by the prior term
for name, bf in bestfits.items():
    print(f"{name}: freefit llh = {bf['llh']:.4f}")

In [ ]:
scans, errors = {}, {}
for name in scan_dict:
    scans[name], errors[name] = {}, {}
    for parameter in PARAMETERS:
        try:
            scan = scan_dict[name].get_scan_df(param_name=parameter, drop_unsuccessful=drop_unsuccessful)
        except Exception as e:
            print(f"{name}: skipping {parameter}: {e}")
            continue
        bestfit_val = bestfits[name][parameter]
        scans[name][parameter] = scan
        errors[name][parameter] = (bestfit_val, *get_1sigma(scan, parameter, bestfit_val))

def plot_profiles(parameters, ncols, savepath, legend_errors=False):
    nrows = int(np.ceil(len(parameters) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows))
    axes = np.atleast_1d(axes).flatten()
    for ax, parameter in zip(axes, parameters):
        for name in scan_dict:
            if parameter not in scans[name]:
                continue
            scan_sorted = scans[name][parameter].sort_values(parameter)
            bestfit_val, err_lo, err_hi = errors[name][parameter]
            label = (f"{name}: ${bestfit_val:.3f}^{{+{err_hi:.3f}}}_{{-{err_lo:.3f}}}$" if legend_errors
                     else f"{name}: {bestfit_val:.3g}")
            ax.plot(scan_sorted[parameter], scan_sorted["dllh"], color=colors[name], linewidth=2, label=label)
            ax.axvline(bestfit_val, color=colors[name], linewidth=1.5, linestyle="--")
            for edge in [bestfit_val - err_lo, bestfit_val + err_hi]:
                if np.isfinite(edge):
                    ax.axvline(edge, color=colors[name], linestyle=":", linewidth=1.2)
        ax.axhline(1.0, color="black", linestyle="--", linewidth=1.2)
        ax.set_xlabel(parameter, fontdict=font_axis_label)
        ax.set_ylabel(r"$-2\Delta\mathrm{log}\mathcal{L}$", fontdict=font_axis_label)
        ax.set_ylim(-0.5, 5)
        ax.legend(prop=font_legend)
    for ax in axes[len(parameters):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.savefig(savepath, bbox_inches="tight")
    plt.show()

In [ ]:
plot_profiles(["astro_norm", "gamma_astro", "astro_nue_ratio", "astro_nutau_ratio"], ncols=2,
              savepath=f"{plotting_path}/signal_scan.png", legend_errors=True)

In [ ]:
plot_profiles(PARAMETERS, ncols=4, savepath=f"{plotting_path}/all_param_scan.png")

In [ ]:
# best fit +- 1 sigma relative to the SAY best fit
fig, ax = plt.subplots(figsize=(7, 0.65 * len(PARAMETERS) + 2))
ypos = np.arange(len(PARAMETERS))[::-1]
say_bf = np.array([errors["SAY"][p][0] for p in PARAMETERS])
for offset, name in zip([0.2, 0.0, -0.2], scan_dict):
    vals = np.array([errors[name].get(p, (np.nan,) * 3) for p in PARAMETERS])
    ax.errorbar(vals[:, 0] - say_bf, ypos + offset, xerr=[np.nan_to_num(vals[:, 1]), np.nan_to_num(vals[:, 2])],
                fmt="o", color=colors[name], elinewidth=2, capsize=4, markersize=5, label=name)
ax.axvline(0, color="black", linewidth=1.2, linestyle="--")
ax.set_yticks(ypos)
ax.set_yticklabels(PARAMETERS)
ax.set_xlabel("deviation from SAY best fit", fontdict=font_axis_label)
ax.grid(axis="x", alpha=0.3)
ax.legend(prop=font_legend)
plt.tight_layout()
plt.savefig(f"{plotting_path}/errors_overview.png", bbox_inches="tight")
plt.show()

## Best fit table

In [ ]:
df_bestfit = pd.DataFrame({(name, col): {p: v[i] for p, v in errors[name].items()}
                           for name in scan_dict for i, col in enumerate(["bestfit", "-1sig", "+1sig"])}).loc[PARAMETERS]
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200, "display.max_columns", 20):
    display(df_bestfit)

print(latex_comparison_table(errors,
                             caption=r"Step2 best fit values with $1\sigma$ asymmetric uncertainties for the SAY likelihood, "
                                     r"the Poisson likelihood, and the Poisson likelihood with tight detector priors.",
                             label="tab:bestfit_SAY_Poisson_tightpriors"))

## Flavor triangle
SAY: the three scans `nbestfit50_fine_{0,1,2}` combined (as in `flavor_triangle.ipynb`).

In [ ]:
sys.path.append("/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks")
from plot_utils_triangle import *

triangle_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor_reunblind/data_fits_flavor_triangle"
tri_scan_dirs = {
    "nbestfit50_fine_0": f"{triangle_path}/nbestfit50_fine_0",
    "nbestfit50_fine_1": f"{triangle_path}/nbestfit50_fine_1",
    "nbestfit50_fine_2": f"{triangle_path}/nbestfit50_fine_2",
    "Poisson":           f"{triangle_path}/nbestfit50_Poisson",
    "Poisson tight":     f"{triangle_path}/nbestfit50_Poisson_tightpriors",
}
N_grid = 40
tri_scans, tri_data, tri_bestfits = {}, {}, {}
for name, scan_dir in tri_scan_dirs.items():
    tri_scans[name], tri_data[name], tri_bestfits[name] = load_triangle_scan(scan_dir, N_grid=N_grid)
    print(f"{name}: {len(tri_data[name]['ts'])} successful scan points, freefit llh = {tri_bestfits[name]['llh']:.4f}")

tri_data["SAY"], tri_bestfits["SAY"] = combine_scans(["nbestfit50_fine_0", "nbestfit50_fine_1", "nbestfit50_fine_2"],
                                                     tri_data, tri_bestfits, N_grid)
tri_names = ["SAY", "Poisson", "Poisson tight"]

In [ ]:
# best fit of the flavor triangle scans (no errors from the 2D scan)
tri_parameters = ["astro_norm", "gamma_astro", "astro_nue_ratio", "astro_nutau_ratio", "f_e", "f_mu", "f_tau"]
tri_results = {name: {**{p: tri_bestfits[name][p] for p in tri_parameters[:4]}, **flavor_fractions(tri_bestfits[name])}
               for name in tri_names}
with pd.option_context("display.float_format", "{:.3f}".format):
    display(pd.DataFrame(tri_results).loc[tri_parameters])

print(latex_comparison_table({name: {p: (v, np.nan, np.nan) for p, v in res.items()} for name, res in tri_results.items()},
                             caption=r"Best fit of the flavor triangle scans for the SAY likelihood, the Poisson likelihood, "
                                     r"and the Poisson likelihood with tight detector priors.",
                             label="tab:flavor_triangle_SAY_Poisson_tightpriors",
                             parameters=tri_parameters))

In [ ]:
fr = flavor_fractions(tri_bestfits["Poisson tight"])
C = plot_contour(data=tri_data, name="Poisson tight", label="Poisson, tight priors", levels=["68%", "90%", "95%"], title="",
                 scenarios=True, show_ts_labels=False, bestfit=[fr["f_tau"], fr["f_e"]],
                 savepath=f"{plotting_path}/flavor_triangle_Poisson_tightpriors.png")
plt.show()

In [ ]:
C = compare_contours(tri_data, names=tri_names, labels=["SAY (3 scans combined)", "Poisson", "Poisson, tight priors"],
                     levels=["68%", "90%", "95%"], title="", scenarios=True)
ax = plt.gca()
for i, name in enumerate(tri_names):
    fr = flavor_fractions(tri_bestfits[name])
    ax.scatter([fr["f_tau"]], [fr["f_e"]], marker="X", facecolor=colours[i], edgecolor="k", lw=0.5, s=80, zorder=10)
plt.savefig(f"{plotting_path}/flavor_triangle_SAY_Poisson_tight.png", bbox_inches="tight")
plt.show()